# Project 2 figures
Run from top to bottom. The first code cell contains the font settings. Each plot cell contains its actual plotting commands, `figsize`, spacing and legend placement; change these directly and rerun the cell. Every saved figure is displayed first. PDFs are written beside this notebook and copied to the sibling thesis repository.

The original paper typography and per-figure canvas sizes are restored. Three-panel rows and fig2.2 use the requested 2×2 arrangement, with their shared legend in the fourth quadrant. The verified scientific corrections remain in place. No first-principles calculation is run.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, shutil, logging, json, csv, re, io, os, subprocess, hashlib
sys.dont_write_bytecode = True
import xml.etree.ElementTree as ET
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from PIL import Image
import fitz, h5py, yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'figures_for_thesis':
    ROOT = ROOT.parent
if not (ROOT / 'figures_for_thesis').is_dir():
    raise FileNotFoundError('Run this notebook from the project root or figures_for_thesis directory.')
OUT = ROOT / 'figures_for_thesis'
THESIS = ROOT.parent / 'PhD_thesis_20251216' / 'figures_proj2'
sys.path.insert(0, str(ROOT))
logging.getLogger('fontTools').setLevel(logging.ERROR)

# Shared paper fonts. Override an individual artist's fontsize in its plot cell if needed.
BLUE, GREEN, YELLOW, ORANGE = '#1478E1', '#28AF3C', '#FAC828', '#FA8C00'
PURPLE, CYAN, GREY = '#8C64E1', '#32B4C8', '#787878'
plt.rcParams.update({'text.usetex': False, 'font.family': 'serif', 'mathtext.fontset': 'cm',
 'axes.titlesize': 20, 'axes.labelsize': 16, 'xtick.labelsize': 14, 'ytick.labelsize': 14,
 'legend.fontsize': 12, 'figure.dpi': 196, 'figure.facecolor': 'w',
 'lines.linewidth': 1.5, 'lines.solid_capstyle': 'round', 'lines.dash_capstyle': 'round',
 'lines.solid_joinstyle': 'round', 'lines.dash_joinstyle': 'round', 'pdf.fonttype': 42})


Saving and scientific data readers

In [ ]:
def frame(ax):
    ax.tick_params(direction='in', which='both', top=True, right=True)

def save(fig, name):
    display(fig)
    path = OUT / name
    fig.savefig(path, metadata={'CreationDate': None})
    shutil.copy2(path, THESIS / name)
    plt.close(fig)
    print(name)

from vmatplot.bandstructure import extract_kpath, kpoints_path_lists
from vmatplot.dos import (_read_outcar_parameters, _read_outcar_eigen_matrices,
                          _read_outcar_kpoint_weights)

@lru_cache(None)
def bands(folder):
    directory = ROOT / '3.1_bandstructure' / folder
    root = ET.parse(directory / 'vasprun.xml').getroot()
    if (directory / 'KPOINTS_OPT').exists():
        eigen = root.find('./calculation/eigenvalues_kpoints_opt/eigenvalues')
        fermi = float(root.find("./calculation/dos[@comment='kpoints_opt']/i[@name='efermi']").text)
    else:
        eigen = root.find('./calculation/eigenvalues')
        fermi = float(root.find(".//i[@name='efermi']").text)
    energies = []
    for spin in eigen.findall('./array/set/set'):
        energies.append(np.array([[float(r.text.split()[0]) for r in k.findall('r')]
                                   for k in spin.findall('set')]) - fermi)
    x, breaks = extract_kpath(str(directory), return_breaks=True)
    ticks, labels = kpoints_path_lists(str(directory))
    return np.array(x), energies, ticks, labels, breaks, fermi

def draw_bands(ax, folder, spin=False, color=BLUE, label=None, linestyle='-'):
    x, energies, ticks, labels, breaks, _ = bands(folder)
    for channel, values in enumerate(energies if spin else energies[:1]):
        xx, yy = x.copy(), values.copy()
        for i in reversed(breaks):
            xx = np.insert(xx, i, xx[i]); yy = np.insert(yy, i, np.nan, axis=0)
        line = ax.plot(xx, yy, color=(PURPLE, CYAN)[channel] if spin else color,
                       ls=('-', (0,(4,3)))[channel] if spin else linestyle)
        if label: line[0].set_label(label)
    for tick in ticks[1:-1]: ax.axvline(tick, color=GREY, ls='--', alpha=.8, zorder=0)
    ax.axhline(0, color='#5A3C8C', ls='--', alpha=.8, zorder=1)
    ax.set_xticks(ticks, labels); ax.set_xlim(x[0], x[-1]); ax.set_ylim(-4, 3)
    frame(ax)

def spin_legend(fig, ax, bilayer=False):
    handles = [Line2D([],[],color=PURPLE,label='Spin up'),
               Line2D([],[],color=CYAN,ls=(0,(4,3)),label='Spin down')]
    if bilayer: handles.append(Line2D([],[],color=BLUE,label='Bilayer bands'))
    handles.append(Line2D([],[],color=GREY,ls='--',label=r'$E_{\mathrm{F}}=0$'))
    ax.legend(handles=handles,loc='center',frameon=True,fancybox=True,
              borderpad=.3,labelspacing=.35,handlelength=1.5)

@lru_cache(None)
def dos(folder):
    directory = ROOT / '4.1_PDoS' / folder
    if folder.endswith('_ollie'):
        # Both channels must be reconstructed on the same absolute energy grid.
        lines=(directory/'OUTCAR').read_text().splitlines()
        params=_read_outcar_parameters(lines)
        eigen,_=_read_outcar_eigen_matrices(lines)
        sigma=params['sigma']; assert sigma == .05
        assert eigen[1].shape == eigen[2].shape == (32,25)
        weights=_read_outcar_kpoint_weights(lines,25)
        lo=min(a.min() for a in eigen.values())-6*sigma
        hi=max(a.max() for a in eigen.values())+6*sigma
        energy=np.linspace(lo,hi,params['nedos'])
        channels=[]
        for spin in (1,2):
            values=np.zeros_like(energy)
            for k,weight in enumerate(weights):
                z=(energy[:,None]-eigen[spin][:,k])/sigma
                values += weight*np.exp(-.5*z*z).sum(axis=1)/(sigma*np.sqrt(2*np.pi))
            channels.append(values)
        return energy-params['efermi'],channels,params['efermi']
    root = ET.parse(directory / 'vasprun.xml').getroot()
    section = root.find('.//dos')
    fermi = float(section.find("i[@name='efermi']").text)
    arrays = [np.array([[float(v) for v in r.text.split()] for r in spin.findall('r')])
              for spin in section.findall('./total/array/set/set')]
    return arrays[0][:,0] - fermi, [a[:,1] for a in arrays], fermi


Three original 9 x 6 panels, arranged as 2 x 2 with one shared legend.

In [ ]:
fig, grid = plt.subplots(2, 2, figsize=(18, 12))
for ax, folder, title in zip(grid.flat[:3],
        ['monolayer_FM_HSE06', 'monolayer_AFM_HSE06', 'bilayer_HSE'],
        ['(a) Band structure for FM monolayer o-B$_{14}$',
         '(b) Band structure for AFM monolayer o-B$_{14}$',
         '(c) Band structure for bilayer o-B$_{14}$']):
    draw_bands(ax, folder, spin=folder!='bilayer_HSE')
    ax.set_ylabel('Energy (eV)'); ax.set_title(title)
grid[1, 1].axis('off')
handles = [Line2D([], [], color=PURPLE, label='Spin up'),
           Line2D([], [], color=CYAN, ls=(0, (4, 3)), label='Spin down'),
           Line2D([], [], color=BLUE, label='Bilayer bands'),
           Line2D([], [], color='#5A3C8C', ls='--', label='Fermi energy')]
grid[1, 1].legend(handles=handles, loc='center')
fig.subplots_adjust(left=.07, right=.97, bottom=.06, top=.94, wspace=.18, hspace=.22)
save(fig, 'fig2.6.pdf')


Original 9 x 6 panels, side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, folder, method in zip(axes, ['monolayer_FM', 'monolayer_FM_HSE06'], ['GGA-PBE', 'HSE06']):
    draw_bands(ax, folder, spin=True)
    ax.set_ylabel('Energy (eV)')
    ax.set_title('Band structure for FM monolayer o-B$_{14}$')
    ax.text(.03, .96, method, transform=ax.transAxes, va='top', fontsize=16,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=.75,
                      edgecolor='#B4B4B4', linewidth=1.5))
    ax.legend(handles=handles[:2]+handles[-1:], loc='upper right')
fig.subplots_adjust(left=.06, right=.97, bottom=.10, top=.88, wspace=.18)
save(fig, 'S2.11.pdf')


S2.13.pdf — Original single-plot canvases and internal legends.

In [ ]:
(filename, folders, labels, title) = ('S2.13.pdf', ['monolayer', 'monolayer_HSE', 'monolayer_R2SCAN'], ['GGA-PBE', 'HSE06', 'R2SCAN'], 'Band structure for monolayer o-B$_{14}$')
fig, ax = plt.subplots(figsize=(10, 6))
for folder, label, color in zip(folders, labels, [BLUE, ORANGE, '#8CAF28']):
    draw_bands(ax, folder, color=color, label=label)
ax.set_ylim(-4, 4)
ax.set_ylabel('Energy (eV)')
ax.set_title(title)
lines, names = ax.get_legend_handles_labels()
ax.legend(lines + handles[-1:], names + ['Fermi energy'], loc='upper right')
fig.subplots_adjust(left=0.1, right=0.97, bottom=0.1, top=0.89)
save(fig, filename)


S2.14.pdf — Original single-plot canvases and internal legends.

In [ ]:
(filename, folders, labels, title) = ('S2.14.pdf', ['monolayer', 'monolayer_shifting', 'monolayer_sym_off'], ['Bands of GGA-PBE', 'Bands of shifted atoms', 'Bands with symmetry off'], 'Band structure for monolayer o-B$_{14}$ for symmetry testing')
fig, ax = plt.subplots(figsize=(10, 6))
for folder, label, color in zip(folders, labels, [BLUE, ORANGE, '#8CAF28']):
    draw_bands(ax, folder, color=color, label=label)
ax.set_ylim(-4, 4)
ax.set_ylabel('Energy (eV)')
ax.set_title(title)
lines, names = ax.get_legend_handles_labels()
ax.legend(lines + handles[-1:], names + ['Fermi energy'], loc='upper right')
fig.subplots_adjust(left=0.1, right=0.97, bottom=0.1, top=0.89)
save(fig, filename)


Original 12 x 6 FM and AFM plots, stacked without equalizing their printed font size.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 12))
for ax, folder, order in zip(axes, ['monolayer_FM_ollie', 'monolayer_AFM_ollie'], ['FM', 'AFM']):
    energy, channels, _ = dos(folder)
    ax.plot(energy, channels[0]+channels[1], color=BLUE, label='Total')
    ax.plot(energy, channels[0], color=ORANGE, label='Spin up')
    ax.plot(energy, -channels[1], color=CYAN, label='Spin down')
    ax.axvline(0, color='#5A3C8C', ls='--')
    ax.set(xlim=(-14, 6), ylim=(-8, 15), xlabel='Energy (eV)',
           ylabel='Density of States (states/eV)',
           title=f'Spin-polarized DoS of {order} monolayer o-B$_{{14}}$')
    ax.legend(loc='upper right'); frame(ax)
fig.subplots_adjust(left=.10, right=.97, bottom=.06, top=.95, hspace=.30)
save(fig, 'S2.16.pdf')


Total DOS, original 10 x 6 canvas.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for folder, label, color in [('o-B14_K20', 'Bulk', BLUE), ('monolayer', 'Monolayer', GREEN),
        ('bilayer', 'Bilayer', YELLOW), ('bilayer_with_Hydrogen', 'H-terminated bilayer', ORANGE)]:
    energy, channels, _ = dos(folder)
    ax.plot(energy, channels[0], color=color, label=label)
ax.axvline(0, color='#5A3C8C', ls='--')
ax.set(xlim=(-6, 6), ylim=(0, 27), xlabel='Energy (eV)', ylabel='Density of States',
       title='Total DoS for o-B$_{14}$ systems')
frame(ax); ax.legend(loc='upper right')
fig.subplots_adjust(left=.10, right=.97, bottom=.13, top=.89)
save(fig, 'S2.12.pdf')


Original 12 x 6 band/DOS layout with 3:1 widths.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [3, 1]}, sharey=True)
draw_bands(axes[0], 'bilayer_with_Hydrogen'); axes[0].set_ylim(-4, 4)
energy, channels, _ = dos('bilayer_with_Hydrogen')
axes[1].plot(channels[0], energy, color=BLUE)
axes[1].axhline(0, color='#5A3C8C', ls='--'); axes[1].set_xlim(0, 20)
axes[0].set_ylabel('Energy (eV)')
axes[0].set_title('Band structure', fontsize=18); axes[1].set_title('DoS (a.u.)', fontsize=18)
frame(axes[1])
fig.suptitle('Band structure and DoS for bilayer o-B$_{14}$ with hydrogen termination', fontsize=20)
fig.subplots_adjust(left=.09, right=.96, bottom=.10, top=.84, wspace=.08)
save(fig, 'fig2.8.pdf')


Optical data and formulas

In [ ]:
systems = [('bilayer_with_Hydrogen', 'H-terminated bilayer', ORANGE, 30.088116646 / 11.208620),
           ('bilayer', 'Bilayer', YELLOW, 28.088116646 / 9.41157),
           ('monolayer', 'Monolayer', GREEN, 24.044058323 / 5.807056948202938),
           ('o-B14_n128_k34', 'Bulk', BLUE, 1.)]


def dielectric(folder, factor=1.):
    with h5py.File(ROOT / '5.1_dielectric_function' / folder / 'vaspout.h5') as f:
        group = f['results/linear_response']
        energy = group['energies_dielectric_function'][:]
        tensor = group['density_density_dielectric_function'][:]
    real = np.array([tensor[i, i, :, 0] for i in range(3)]).T
    imag = np.array([tensor[i, i, :, 1] for i in range(3)]).T
    return energy, 1 + factor * (real - 1), factor * imag


def quantity(energy, real, imag, kind):
    modulus = np.hypot(real, imag)
    n = np.sqrt(np.maximum(0, (modulus + real) / 2))
    k = np.sqrt(np.maximum(0, (modulus - real) / 2))
    return {'real': real, 'imag': imag, 'n': n, 'k': k,
            'alpha': 2 * energy[:, None] * k / (6.582119569e-16 * 2.99792458e18),
            'loss': imag / (real**2 + imag**2),
            'R': ((n - 1)**2 + k**2) / ((n + 1)**2 + k**2)}[kind]



data = {folder: dielectric(folder, factor) for folder, _, _, factor in systems}


Original dielectric tensor figure: three columns, real above imaginary.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
for row, kind in enumerate(('real', 'imag')):
    for col, component in enumerate(('xx', 'yy', 'zz')):
        ax = axes[row, col]
        for folder, label, color, factor in systems:
            energy, real, imag = data[folder]
            chosen = (energy >= 0) & (energy <= 12)
            ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
        if kind == 'real':
            lo, hi = ax.get_ylim(); ax.set_ylim(max(-60, lo), min(60, hi))
        else:
            hi = min(60, ax.get_ylim()[1]); ax.set_ylim(-.05*hi, hi)
        ax.set_xlim(0, 12); frame(ax)
        ax.set_title(f'{"Real" if row == 0 else "Imaginary"} part for {component}-component')
        if col == 0: ax.set_ylabel('Dielectric function', fontsize=20)
        if row == 1: ax.set_xlabel('Photon energy (eV)', fontsize=18)
        ax.legend(loc='best')
fig.suptitle('Dielectric function', fontsize=20)
fig.subplots_adjust(left=.055, right=.985, bottom=.08, top=.91, wspace=.18, hspace=.28)
save(fig, 'fig2.10.pdf')


fig2.11a.pdf — Each original 24 x 6, three-panel optical figure becomes a 16 x 12 grid.

In [ ]:
(name, kind, title, ylabel) = ('fig2.11a.pdf', 'alpha', 'Absorption coefficient', 'Absorption coefficient ($\\mathrm{\\AA}^{-1}$)')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for col, ax in enumerate(axes.flat[:3]):
    for folder, label, color, factor in systems:
        energy, real, imag = data[folder]
        chosen = (energy >= 0) & (energy <= 12)
        ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
    ax.set(xlim=(0, 12), xlabel='Photon energy (eV)', ylabel=ylabel, title=f"({'abc'[col]}) {('xx', 'yy', 'zz')[col]}-component")
    frame(ax)
axes[1, 1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[1, 1].legend(handles, labels, loc='center')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.07, top=0.9, wspace=0.22, hspace=0.25)
save(fig, name)


fig2.11b.pdf — Each original 24 x 6, three-panel optical figure becomes a 16 x 12 grid.

In [ ]:
(name, kind, title, ylabel) = ('fig2.11b.pdf', 'loss', 'Energy-loss spectrum', 'Energy-loss spectrum')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for col, ax in enumerate(axes.flat[:3]):
    for folder, label, color, factor in systems:
        energy, real, imag = data[folder]
        chosen = (energy >= 0) & (energy <= 12)
        ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
    ax.set(xlim=(0, 12), xlabel='Photon energy (eV)', ylabel=ylabel, title=f"({'abc'[col]}) {('xx', 'yy', 'zz')[col]}-component")
    frame(ax)
axes[1, 1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[1, 1].legend(handles, labels, loc='center')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.07, top=0.9, wspace=0.22, hspace=0.25)
save(fig, name)


fig2.12a.pdf — Each original 24 x 6, three-panel optical figure becomes a 16 x 12 grid.

In [ ]:
(name, kind, title, ylabel) = ('fig2.12a.pdf', 'R', 'Reflectivity', 'Reflectivity')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for col, ax in enumerate(axes.flat[:3]):
    for folder, label, color, factor in systems:
        energy, real, imag = data[folder]
        chosen = (energy >= 0) & (energy <= 12)
        ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
    ax.set(xlim=(0, 12), xlabel='Photon energy (eV)', ylabel=ylabel, title=f"({'abc'[col]}) {('xx', 'yy', 'zz')[col]}-component")
    frame(ax)
axes[1, 1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[1, 1].legend(handles, labels, loc='center')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.07, top=0.9, wspace=0.22, hspace=0.25)
save(fig, name)


fig2.12b.pdf — Each original 24 x 6, three-panel optical figure becomes a 16 x 12 grid.

In [ ]:
(name, kind, title, ylabel) = ('fig2.12b.pdf', 'n', 'Refractive index', 'Refractive index')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for col, ax in enumerate(axes.flat[:3]):
    for folder, label, color, factor in systems:
        energy, real, imag = data[folder]
        chosen = (energy >= 0) & (energy <= 12)
        ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
    ax.set(xlim=(0, 12), xlabel='Photon energy (eV)', ylabel=ylabel, title=f"({'abc'[col]}) {('xx', 'yy', 'zz')[col]}-component")
    frame(ax)
axes[1, 1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[1, 1].legend(handles, labels, loc='center')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.07, top=0.9, wspace=0.22, hspace=0.25)
save(fig, name)


S2.18.pdf — Each original 24 x 6, three-panel optical figure becomes a 16 x 12 grid.

In [ ]:
(name, kind, title, ylabel) = ('S2.18.pdf', 'k', 'Extinction coefficient', 'Extinction coefficient')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for col, ax in enumerate(axes.flat[:3]):
    for folder, label, color, factor in systems:
        energy, real, imag = data[folder]
        chosen = (energy >= 0) & (energy <= 12)
        ax.plot(energy[chosen], quantity(energy, real, imag, kind)[chosen, col], color=color, label=label)
    ax.set(xlim=(0, 12), xlabel='Photon energy (eV)', ylabel=ylabel, title=f"({'abc'[col]}) {('xx', 'yy', 'zz')[col]}-component")
    frame(ax)
axes[1, 1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
axes[1, 1].legend(handles, labels, loc='center')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.08, right=0.97, bottom=0.07, top=0.9, wspace=0.22, hspace=0.25)
save(fig, name)


S2.4.pdf — Original 24 x 12 dielectric convergence figures.

In [ ]:
(name, folders, labels, limits, title) = ('S2.4.pdf', [f'o-B14_n128_k{k}' for k in (10, 20, 26, 30, 32, 34)], ['10×14×9', '20×28×18', '26×37×24', '30×42×28', '32×45×29', '34×48×31'], [(1, 8), (5, 15), (0, 4)], 'Dielectric function versus k-points for bulk o-B$_{14}$')
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
for folder, label, color in zip(folders, labels, [GREEN, '#E65050', ORANGE, YELLOW, CYAN, BLUE]):
    energy, real, imag = dielectric(folder)
    for row, values in enumerate((real, imag)):
        for col in range(3):
            axes[row, col].plot(energy, values[:, col], color=color, label=label)
for i, ax in enumerate(axes.flat):
    row, col = divmod(i, 3)
    ax.set_xlim(limits[col])
    frame(ax)
    ax.set_title(f"{('Real' if row == 0 else 'Imaginary')} part for {('xx', 'yy', 'zz')[col]}-component")
    if col == 0:
        ax.set_ylabel('Dielectric function', fontsize=20)
    if row == 1:
        ax.set_xlabel('Photon energy (eV)', fontsize=18)
    shown = [line.get_ydata()[(line.get_xdata() >= limits[col][0]) & (line.get_xdata() <= limits[col][1])] for line in ax.lines]
    lo, hi = (min((a.min() for a in shown)), max((a.max() for a in shown)))
    ax.set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.legend(loc='best')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.055, right=0.985, bottom=0.08, top=0.91, wspace=0.18, hspace=0.28)
save(fig, name)


S2.5.pdf — Original 24 x 12 dielectric convergence figures.

In [ ]:
(name, folders, labels, limits, title) = ('S2.5.pdf', [f'o-B14_n{k}_k10' for k in (32, 64, 128, 256)], ['NBANDS = 48', 'NBANDS = 72', 'NBANDS = 144', 'NBANDS = 264'], [(15, 25), (15, 30), (10, 30)], 'Dielectric function versus NBANDS for bulk o-B$_{14}$')
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
for folder, label, color in zip(folders, labels, [GREEN, '#E65050', ORANGE, YELLOW, CYAN, BLUE]):
    energy, real, imag = dielectric(folder)
    for row, values in enumerate((real, imag)):
        for col in range(3):
            axes[row, col].plot(energy, values[:, col], color=color, label=label)
for i, ax in enumerate(axes.flat):
    row, col = divmod(i, 3)
    ax.set_xlim(limits[col])
    frame(ax)
    ax.set_title(f"{('Real' if row == 0 else 'Imaginary')} part for {('xx', 'yy', 'zz')[col]}-component")
    if col == 0:
        ax.set_ylabel('Dielectric function', fontsize=20)
    if row == 1:
        ax.set_xlabel('Photon energy (eV)', fontsize=18)
    shown = [line.get_ydata()[(line.get_xdata() >= limits[col][0]) & (line.get_xdata() <= limits[col][1])] for line in ax.lines]
    lo, hi = (min((a.min() for a in shown)), max((a.max() for a in shown)))
    ax.set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    ax.legend(loc='best')
fig.suptitle(title, fontsize=20)
fig.subplots_adjust(left=0.055, right=0.985, bottom=0.08, top=0.91, wspace=0.18, hspace=0.28)
save(fig, name)


Phonon data

In [ ]:
from vmatplot.phonon import extract_phonon_bands, extract_qpath


@lru_cache(None)
def phonon(folder):
    directory=ROOT/folder
    if (directory/'band.yaml').exists():
        with open(directory/'band.yaml') as f: raw=yaml.load(f,Loader=yaml.CSafeLoader)
        x=np.array([q['distance'] for q in raw['phonon']])
        y=np.array([[b['frequency'] for b in q['band']] for q in raw['phonon']])
        assert np.all(np.diff(x)>=-1e-8)
        ends=np.cumsum(raw['segment_nqpoint'])
        ticks=[x[0]]
        for end in ends:
            if x[end-1]>ticks[-1]+1e-10: ticks.append(x[end-1])
        labels=['Γ','Z','T','Y','Γ']; assert len(ticks)==len(labels)
        for end in reversed(ends[:-1]):
            x=np.insert(x,end,np.nan); y=np.insert(y,end,np.nan,axis=0)
        return x,y,ticks,labels
    else:
        raw=extract_phonon_bands(str(directory)); x=np.array(extract_qpath(str(directory)))
        n=len(x)//4
        ticks=x[[0,n-1,2*n-1,3*n-1,len(x)-1]]; labels=['Γ','Z','T','Y','Γ']
    return x, np.array(raw['bands']).T, ticks, labels


def draw_phonon(ax,folder,label,color):
    x,y,ticks,labels=phonon(folder)
    lines=ax.plot(x,y,color=color); lines[0].set_label(label)
    ax.set_xticks(ticks,labels); ax.set_xlim(np.nanmin(x),np.nanmax(x))
    frame(ax)


pristine=[('3.0_phonon_dispersion_vasp/monolayer_3','Monolayer',CYAN),
          ('3.0_phonon_dispersion_vasp/bilayer_3','Bilayer',YELLOW)]
hydrogen=[('3.0_phonon_dispersion_phononpy/monolayer_top','H-terminated monolayer','#8CAF28'),
          ('3.0_phonon_dispersion_phononpy/bilayer_top','H-terminated bilayer',ORANGE)]


Original two 9 x 6 phonon plots, side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, group in zip(axes, [pristine, hydrogen]):
    for folder, label, color in group: draw_phonon(ax, folder, label+' 3×3', color)
    ax.axhline(0, color='#5A3C8C', ls='--', zorder=0)
    ax.set(ylim=(-3.3, 8), ylabel='Frequency (THz)', title='Phonon dispersion')
    ax.legend(loc='upper right')
fig.subplots_adjust(left=.06, right=.97, bottom=.10, top=.88, wspace=.18)
save(fig, 'S2.9.pdf')


fig2.5.pdf — Original single phonon figures.

In [ ]:
(filename, group, ylim) = ('fig2.5.pdf', pristine, (-1, 7))
fig, ax = plt.subplots(figsize=(10, 6))
for folder, label, color in group:
    draw_phonon(ax, folder, label, color)
ax.axhline(0, color='#5A3C8C', ls='--', zorder=0)
ax.set(ylim=ylim, ylabel='Frequency (THz)', title='Phonon dispersion')
ax.legend(loc='upper right')
fig.subplots_adjust(left=0.1, right=0.97, bottom=0.1, top=0.89)
save(fig, filename)


S2.8.pdf — Original single phonon figures.

In [ ]:
(filename, group, ylim) = ('S2.8.pdf', [(f'3.0_phonon_dispersion_vasp/{folder}', label + ' 2×2', color) for folder, label, color in [('monolayer_2', 'Monolayer', CYAN), ('monolayer_H_2', 'H-terminated monolayer', '#8CAF28'), ('bilayer_2', 'Bilayer', YELLOW), ('bilayer_H_2', 'H-terminated bilayer', ORANGE)]], (-3, 4))
fig, ax = plt.subplots(figsize=(10, 6))
for folder, label, color in group:
    draw_phonon(ax, folder, label, color)
ax.axhline(0, color='#5A3C8C', ls='--', zorder=0)
ax.set(ylim=ylim, ylabel='Frequency (THz)', title='Phonon dispersion')
ax.legend(loc='upper right')
fig.subplots_adjust(left=0.1, right=0.97, bottom=0.1, top=0.89)
save(fig, filename)


fig2.9 — original gap canvases, corrected archived distributions

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))
cmap=LinearSegmentedColormap.from_list('gap_density',['white',BLUE])
records={}
for ax,folder,title in zip(axes,['bulk','bilayer_with_Hydrogen'],
                          ['(a) Bulk o-B$_{14}$','(b) H-terminated bilayer o-B$_{14}$']):
    rows=[]
    for file in sorted((ROOT/'superconductivity'/folder).glob('B14.imag_aniso_gap0_*')):
        temperature=float(file.name.rsplit('_',1)[1])
        data=np.loadtxt(file,ndmin=2)
        density=data[:,0]-temperature
        delta=data[:,1]
        assert np.isfinite(data).all() and np.all(density>=0)
        assert np.isclose(density.max(),1) and np.all(density<=1+1e-8)
        # Column 1 is T + rho/max(rho), not an independent temperature sample.
        points=ax.scatter(np.full(len(delta),temperature),delta,c=density,
                          cmap=cmap,vmin=0,vmax=1,marker='_',s=14,linewidths=.9)
        rows.append({'temperature_K':temperature,'bins':len(delta),
                     'gap_min_meV':float(delta.min()),'gap_max_meV':float(delta.max()),
                     'density_mean_meV':float(np.average(delta,weights=density)),
                     'density_min':float(density.min()),'density_max':float(density.max())})
    ax.plot([r['temperature_K'] for r in rows],[r['density_mean_meV'] for r in rows],
            color=ORANGE,label='Distribution mean')
    ax.set(xlim=(0,31),ylim=(0,6.6),xlabel='Temperature (K)')
    ax.set_title(title); ax.set_ylabel(r'Superconducting gap $\Delta$ (meV)'); frame(ax)
    ax.legend(loc='upper right')
    records[folder]=rows

fig.subplots_adjust(left=.065,right=.90,bottom=.14,top=.87,wspace=.20)
colorbar=fig.colorbar(points,cax=fig.add_axes([.925,.14,.020,.73]),
                     orientation='vertical',ticks=[0,.5,1])
colorbar.set_label('Normalized gap density',fontsize=12)
colorbar.ax.tick_params(direction='in',labelsize=12)
save(fig,'fig2.9.pdf')
(OUT/'gap_summary.json').write_text(json.dumps(records,indent=2)+'\n')


Convergence data

In [ ]:
from vmatplot.algorithms import fit_birch_murnaghan
base=ROOT/'1.0_energy/o-B14'


def table(folder,name='energy_parameters.dat'):
    with open(base/folder/name) as f: rows=list(csv.DictReader(f,delimiter='\t'))
    aliases={'kpoints(x y z)':'kpoints mesh','encut':'energy cutoff (encut)'}
    return [{aliases.get(k.lower(),k.lower()):v for k,v in row.items()} for row in rows]


def series(folder,xkey='total kpoints',ykey='total energy',lower=40,name='energy_parameters.dat'):
    rows=table(folder,name)
    rows=sorted([r for r in rows if float(r[xkey])>=lower],key=lambda r:float(r[xkey]))
    return np.array([float(r[xkey]) for r in rows]),np.array([float(r[ykey]) for r in rows]),rows


S2.1 — Dual-x convergence plot and lattice scan side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7.5))
ax = axes[0]; upper = ax.twiny()
for cutoff, color in [(450, GREEN), (480, BLUE)]:
    x, y, rows = series(f'energy_kpoints_{cutoff}')
    nk = np.array([int(r['kpoints mesh'].strip('()').split(',')[0]) for r in rows])
    ax.plot(nk, y, 'o-', ms=4, color=color, label=rf'$E_{{\rm cut}}={cutoff}$ eV')
for grid, label, color in [('1260', r'$10\times14\times9$', PURPLE),
                          ('8721', r'$19\times27\times17$', '#D25ADC')]:
    x, y, _ = series(f'energy_encut_{grid}', 'energy cutoff (encut)', lower=400)
    upper.plot(x, y, 'o-', ms=4, color=color, label=label)
ax.set_xlabel(r'Grid index ($n_x$)', color=BLUE)
ax.set_xticks([4, 8, 12, 16, 20, 24]); ax.set_ylabel('Energy (eV)')
upper.set_xlabel('Energy cutoff (eV)', color=PURPLE)
upper.tick_params(direction='in', colors=PURPLE)
ax.tick_params(direction='in', right=True, colors='black')
ax.tick_params(axis='x', colors=BLUE)
ax.ticklabel_format(axis='y', style='plain', useOffset=False)
ax.set_title('(a) Energy for bulk o-B$_{14}$', pad=56)
lines = ax.get_lines()+upper.get_lines()
ax.legend(lines, [line.get_label() for line in lines], loc='upper right')
x, y, _ = series('energy_lattice', 'lattice constant', lower=0)
params, xx, yy = fit_birch_murnaghan(x, y, sample_count=100)
ax = axes[1]
ax.plot(xx, yy, color=BLUE, label='EOS fit')
ax.plot(x, y, 'o', ms=4, color=BLUE, label='Sampled data')
imin = np.argmin(y)
ax.axvline(x[imin], color=GREY, ls='--', label=rf'Minimum: {x[imin]:.5f} $\mathrm{{\AA}}$')
ax.set(xlabel=r'Lattice constant ($\mathrm{\AA}$)', ylabel='Energy (eV)')
ax.set_title('(b) Energy versus lattice constant for bulk o-B$_{14}$', pad=56)
ax.ticklabel_format(axis='both', style='plain', useOffset=False)
frame(ax); ax.legend(loc='upper right')
fig.subplots_adjust(left=.07, right=.97, bottom=.14, top=.82, wspace=.25)
save(fig, 'S2.1.pdf')


Original 10 x 7.5 cohesive-energy canvas.

In [ ]:
fig,ax=plt.subplots(figsize=(10,7.5))
upper=ax.twiny()
for axis,folder,xkey,color,lower,label in [
 (ax,'energy_kpoints_480','total kpoints',BLUE,40,r'k-grid, $E_{\rm cut}=480$ eV'),
 (upper,'energy_encut_1260','energy cutoff (encut)',PURPLE,400,r'Cutoff, $10\times14\times9$')]:
    rows=table(folder,'cohesive_energy.dat')
    ykey=next(k for k in rows[0] if 'cohesive' in k.lower())
    x,y,rows=series(folder,xkey,ykey,lower,'cohesive_energy.dat')
    if xkey=='total kpoints': x=np.array([int(r['kpoints mesh'].strip('()').split(',')[0]) for r in rows])
    axis.plot(x,y,'o-',ms=4,color=color,label=label)
    axis.ticklabel_format(axis='y',style='plain',useOffset=False)
ax.tick_params(direction='in',right=True,top=False)
upper.tick_params(axis='x',direction='in',colors=PURPLE)
ax.tick_params(axis='x',colors=BLUE)
ax.set_xticks([4,8,12,16,20,24]);ax.set_xlabel(r'Grid index ($n_x$)',color=BLUE)
upper.set_xticks([400,600,800,1000,1200]);upper.set_xlabel('Energy cutoff (eV)',color=PURPLE)
ax.set_ylabel('Cohesive energy (eV/atom)')
handles=ax.get_lines()+upper.get_lines()
ax.legend(handles,[line.get_label() for line in handles],loc='upper right',
           frameon=True,fancybox=True,borderpad=.25,labelspacing=.25)
ax.set_title('Cohesive energy for bulk o-B$_{14}$', pad=56)
fig.subplots_adjust(left=.13,right=.97,bottom=.13,top=.79)
save(fig,'S2.2.pdf')


Original single energy-versus-height canvas.

In [ ]:
with open(ROOT/'2.1_geometry_optimization/monolayer/energy_vacuum/energy_parameters.dat') as f:
    rows=list(csv.DictReader(f,delimiter='\t'))
x=np.array([float(r['a3']) for r in rows]); y=np.array([float(r['total energy']) for r in rows])
fig,ax=plt.subplots(figsize=(10,6));ax.plot(x,y,'o-',color=BLUE,ms=4)
ax.set(xlabel=r'Supercell height ($a_3$, $\mathrm{\AA}$)',ylabel='Energy (eV)')
ax.ticklabel_format(axis='y',style='plain',useOffset=False);frame(ax)
ax.set_title('Energy versus supercell height for monolayer o-B$_{14}$')
fig.subplots_adjust(left=.13,right=.97,bottom=.14,top=.89);save(fig,'S2.3.pdf')


Fixed-spin energies.

In [ ]:
# This supplied tabulation is the source of the thesis fixed-spin figure.
with open(OUT/'fixed_spin_data.csv') as f: rows=list(csv.DictReader(f))
fig,ax=plt.subplots(figsize=(10,6))
for method,color,label in [('PBE+D3',BLUE,'PBE+D3'),('r2SCAN',ORANGE,r'r$^2$SCAN')]:
    selected=[r for r in rows if r['method']==method]
    ax.plot([float(r['fixed_spin_muB']) for r in selected],
            [float(r['relative_energy_eV']) for r in selected],'o-',ms=4,color=color,label=label)
ax.set(xlabel=r'Fixed spin moment ($\mu_{\mathrm{B}}$)',ylabel='Relative energy (eV)')
frame(ax);ax.legend(loc='upper left',ncol=1,frameon=True,fancybox=True)
ax.set_title('Fixed-spin energy profile')
fig.subplots_adjust(left=.11,right=.97,bottom=.14,top=.89)
save(fig,'S2.15.pdf')


fig2.2 — Bands, Brillouin zone and PDoS in a 2×2 grid; columns 2:1, shared legend in the fourth cell.

In [ ]:
# %% Bands, Brillouin zone, PDoS and a shared legend.
fig, axes = plt.subplots(2, 2, figsize=(15, 13.2),
                         gridspec_kw={'width_ratios': [2, 1]})
ax = axes[0, 0]
draw_bands(ax, 'o-B14_K48')
ax.set(ylim=(-4, 4), ylabel='Energy (eV)', title='(a) Band structure for bulk o-B$_{14}$')
with fitz.open(ROOT/'kpath_tide.pdf') as bz_pdf:
    pixels = bz_pdf[0].get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
axes[0, 1].imshow(np.frombuffer(pixels.samples, dtype=np.uint8).reshape(
    pixels.height, pixels.width, 3))
axes[0, 1].axis('off')
axes[0, 1].set_title('(b) Brillouin zone')
r = ET.parse(ROOT/'4.1_PDoS/o-B14_K20/vasprun.xml').getroot(); section = r.find('.//dos')
fermi = float(section.find("i[@name='efermi']").text)
arrays = np.array([[[float(v) for v in row.text.split()] for row in ion.findall('./set/r')]
                   for ion in section.findall('./partial/array/set/set')])
energy = arrays[0, :, 0]-fermi
ax = axes[1, 0]
for group, atoms, color in [(2, range(4,8), BLUE), (3, [0,1,2,3,12,13], ORANGE), (7, range(14), PURPLE)]:
    values = arrays[list(atoms)].sum(axis=0)
    ax.plot(energy, values[:,2:5].sum(axis=1), color=color, label=rf'$p$ for G{group}')
    ax.plot(energy, values[:,1], color=color, ls='--', label=rf'$s$ for G{group}')
ax.axvline(0, color='#5A3C8C', ls='--', label='Fermi energy')
ax.set(xlim=(-6,6), ylim=(0,6), xlabel='Energy (eV)', ylabel='Density of States',
       title='(c) PDoS for bulk o-B$_{14}$')
frame(ax)
axes[1, 1].axis('off')
handles, labels = ax.get_legend_handles_labels()
handles.insert(0, Line2D([], [], color=BLUE, label='Bulk bands'))
labels.insert(0, 'Bulk bands')
axes[1, 1].legend(handles, labels, loc='center', frameon=True)
fig.subplots_adjust(left=.08, right=.98, bottom=.07, top=.94, wspace=.18, hspace=.24)
save(fig, 'fig2.2.pdf')


Original 10 x 6 AIMD curves with the original atomic views on the right.

In [ ]:
rows=[]
for line in (ROOT/'6.0_AIMD/bilayer_H/OSZICAR').read_text().splitlines():
    m=re.search(r'^\s*(\d+)\s+T=\s*([\d.Ee+\-]+).*?F=\s*([\d.Ee+\-]+)',line)
    if m:rows.append([float(x) for x in m.groups()])
step,temp,energy=np.array(rows).T;assert len(step)==5500
fig=plt.figure(figsize=(14,6));gs=fig.add_gridspec(2,2,width_ratios=[2.6,1],left=.09,right=.98,bottom=.13,top=.84,hspace=.13,wspace=.07)
axes=[fig.add_subplot(gs[i,0]) for i in range(2)]
axes[0].plot(step/1000,energy,color=BLUE);axes[1].plot(step/1000,temp,color='#19A0A0')
axes[0].set_ylabel('Electronic free\nenergy (eV)');axes[1].set_ylabel('Temperature (K)')
axes[0].tick_params(labelbottom=False);axes[1].set_xlabel('Time (ps)');axes[1].set_ylim(0,800)
for ax in axes:ax.set_xlim(0,5.5);frame(ax)
for row,filename,title in [(0,'S2.10b1.png','Top\nview'),(1,'S2.10b2.png','Side\nview')]:
    ax=fig.add_subplot(gs[row,1]);ax.imshow(plt.imread(ROOT/'figures_collection'/filename));ax.axis('off');ax.set_title(title.replace('\n',' '),fontsize=18)
fig.suptitle('AIMD simulation for H-terminated bilayer o-B$_{14}$',fontsize=20)
save(fig,'S2.10.pdf')


Original labelled structural artwork — fig2.1, fig2.3, fig2.4, fig2.7 and S2.17
The thesis includes the original high-resolution PNG files directly. The previews below do not change the source pixels.

In [ ]:
# %% Original structural artwork, included in the thesis without PDF conversion.
artwork = [
    ('fig2.1', THESIS/'fig2.1/fig2.1_6k.png', 8),
    ('fig2.3', THESIS/'fig2.3/fig2.3_4k.png', 8),
    ('fig2.4', THESIS/'fig2.4/fig2.4_6k.png', 8),
    ('fig2.7', THESIS/'fig2.7/fig2.7_from2.6.png', 7.5),
    ('S2.17', THESIS/'S2.17/S2.17.png', 7.5),
]
manifest = {}
for name, path, width in artwork:
    with Image.open(path) as pixels:
        dimensions = list(pixels.size)
    manifest[name] = {'source': str(path),
                      'source_sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                      'pixel_dimensions': dimensions,
                      'thesis_width_fraction': width/10,
                      'method': 'Original labelled PNG included directly; no conversion.'}
(OUT/'structure_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
for name, path, width in artwork:
    with Image.open(path) as pixels:
        preview = pixels.copy()
    preview.thumbnail((900, 900))
    display(preview)


Refresh the output manifest
This records source hashes, PDF canvas dimensions, text bounds and identity of the thesis copies. Scientific verification results already recorded in `spin_dos_verification.json` are retained.

In [ ]:
result = subprocess.run([sys.executable, str(OUT / 'provenance.py')], check=True,
                        capture_output=True, text=True,
                        env=dict(os.environ, PYTHONDONTWRITEBYTECODE='1'))
print(result.stdout.strip())
